# Accessing Claude with the API - MODULE 1

Notebook with exemples from Anthropic course `Building with Claude API`.

> Require `pip install "anthropic<1" python-dotenv ipykernel` and one `.env` file with `ANTHROPIC_API_KEY`.

## Setup

In [5]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

## 1. Making a request

In [6]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

print(message.content[0].text)

Quantum computing uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing quantum computers to process certain types of problems exponentially faster than classical computers.


## 2. Multi-turn conversations

In [7]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    print("<< " + message.content[0].text + " >>")
    return message.content[0].text

In [8]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

<< Quantum computing uses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, allowing certain problems to be solved exponentially faster. >>
<< Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously, enabling quantum computers to explore many possible solutions in parallel. >>


## 3. System prompts

In [9]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

messages = [
    {"role": "user", "content": "How do I solve 5x + 2 = 3 for x?"}
]

In [10]:
# Without system prompt
answer = chat(messages)
print(answer)

# Solving 5x + 2 = 3 for x

**Step 1:** Subtract 2 from both sides
- 5x + 2 - 2 = 3 - 2
- 5x = 1

**Step 2:** Divide both sides by 5
- x = 1/5 (or 0.2)

**Check:** 5(1/5) + 2 = 1 + 2 = 3 ✓


In [11]:
# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print(answer)

Great question! Let's work through this step by step.

**Step 1:** What's your goal here?
- You want to get `x` by itself on one side of the equation, right?

**Step 2:** Look at what's happening to `x`:
- You have `5x + 2 = 3`
- The `x` is being multiplied by 5, and then 2 is being added to it
- Which operation should we undo first—the multiplication or the addition?

**Hint:** Think about the order of operations in reverse. We usually do multiplication before addition, so when undoing, we should undo addition before undoing multiplication.

**Step 3:** Try this:
- What do you need to add or subtract from *both sides* to eliminate the `+ 2`?

Once you try that, let me know what you get, and we'll move to the next step!


## 4. Temperature

> `temperature` only works with SDK `anthropic<1`. Starting with version 2.0, the parameter was removed.

In [12]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

messages = [
    {"role": "user", "content": "Give me some movie ideas"}
]

In [13]:
# Low temperature - more predictable
answer = chat(messages, temperature=0.0)
print(answer)

# Movie Ideas

## Drama
- **"The Last Letter"** - A woman discovers unsent letters from her estranged mother and must decide whether to track down the intended recipients
- **"Parallel Lives"** - Two strangers keep accidentally meeting across different cities and time periods, slowly realizing they're connected

## Thriller
- **"The Commute"** - A daily train rider notices the same person watching them; when they finally confront them, they learn something shocking about their own past
- **"Borrowed Time"** - Someone discovers they can see the exact date of death above people's heads—and it's changing

## Comedy
- **"Startup Chaos"** - A group of friends accidentally create a viral app they don't understand and must pretend to be tech geniuses
- **"The Worst Roommate"** - A neat freak and a slob are forced to live together to inherit a house

## Sci-Fi
- **"Frequency Shift"** - Humans discover they can only see one color at a time, and society splits based on which color people choose


In [14]:
# High temperature - more creative
answer = chat(messages, temperature=1.0)
print(answer)

# Movie Ideas

## High Concept
- **The Last Library** - In a future where all books are digital, a hacker discovers a hidden physical library and must protect it from those who want it destroyed
- **Frequency Shift** - A person discovers they can hear radio transmissions from 50 years in the past and tries to prevent a historical tragedy

## Character-Driven
- **The Understudy** - A Broadway understudy finally gets their chance to perform, but realizes the lead actor sabotaged themselves on purpose
- **Wrong Name** - Two strangers keep getting each other's mail and packages, developing a relationship entirely through their mistaken deliveries

## Genre Twists
- **Retirement Protocol** - A heartwarming comedy about a retirement home where residents solve cold cases for fun—until they actually solve one
- **The Commute** - A noir thriller where a detective's daily train ride becomes a puzzle as the same strangers appear, but one detail changes each day

## Lighter Takes
- **Algorithm** -

## 5. Response stream

### Basic Streaming Implementation

To enable streaming, add stream=True to your messages.create call:

In [15]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011Cgt23aMAPL3Xwvbh7cpjn', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), diagnostics=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeDB', type='text_delt

### Simplified Text Streaming

Rather than manually parsing events, you can use the SDK's simplified streaming interface that extracts just the text content:

In [16]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# FakeData Registry

A cloud-based database that generates realistic but entirely fictional personal records, transaction histories, and user profiles for software testing and application development purposes.

### Getting the Complete Message

While streaming individual chunks is great for user experience, you often need the complete message for storage or further processing. After streaming completes, you can get the assembled final message:

In [17]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

In [18]:
print(final_message.content[0].text)

# FakeDB

FakeDB is an in-memory database engine that generates realistic but entirely synthetic data structures for testing and development purposes without requiring actual database infrastructure.


## 6. Structured data

When you need Claude to generate structured data like JSON, Python code, or bulleted lists, you'll often run into a common problem: Claude wants to be helpful and add explanatory text around your content. While this is usually great, sometimes you need just the raw data with nothing else.

### The Problem with Default Responses

By default, when you ask Claude to generate JSON, you might get something like this:

```json
\{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": \{
    "state": ["running"]
  \}
\}
```

This rule captures EC2 instance state changes when instances start running.

### The Solution: Assistant Message Prefilling + Stop Sequences

You can combine assistant message prefilling with stop sequences to get exactly the content you want. Here's how it works:

In [22]:
def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text


In [24]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

print(text)


{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



### Processing the Response

You might notice some extra newline characters in the response. These are easy to handle:

In [25]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())

print(clean_json)

{'Name': 'MySimpleRule', 'EventBusName': 'default', 'EventPattern': {'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification'], 'detail': {'state': ['running']}}, 'State': 'ENABLED', 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction', 'Id': '1'}]}
